# Push to 95% — Reverse Game of Life with Forward-Consistency Training

This notebook demonstrates the process of achieving 95% accuracy in the Reverse Game of Life using forward-consistency training. The approach is memory-optimized for an 8GB RAM CPU-only system.

### Key Innovations:
1. **Differentiable Forward-Consistency Loss**: Predict T-1, run GoL forward, penalize mismatch with input T inside the computation graph.
2. **Multi-Radius Features**: Achieved via `tf.data` pipeline without full preprocessing in RAM.
3. **Per-Cell Classification**: Implemented with dense residual connections.
4. **Iterative Self-Training**: Using confident predictions as extra training data.
5. **Optimal Threshold Search**: Performed during evaluation.

---

In [ ]:
# Import Required Libraries
import os, gc, time, sys
import numpy as np
import pandas as pd
import tensorflow as tf

# Limit TF memory growth
tf.config.threading.set_inter_op_parallelism_threads(2)
tf.config.threading.set_intra_op_parallelism_threads(4)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_NUM_INTEROP_THREADS'] = '2'
os.environ['TF_NUM_INTRAOP_THREADS'] = '4'

# Simple file-based logging with immediate flush
LOG_FILE = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'push_progress.log')
_log_fh = open(LOG_FILE, 'w', buffering=1)  # line-buffered

class _Logger:
    def info(self, msg):
        line = f"{time.strftime('%H:%M:%S')} {msg}"
        print(line, flush=True)
        _log_fh.write(line + '\n')
        _log_fh.flush()
        os.fsync(_log_fh.fileno())

log = _Logger()